# Preprocessing

In [ ]:
# Install libraries and prepare dataset
!pip install transformers datasets scikit-learn numpy


In [ ]:
import pandas as pd

train = pd.read_csv("/content/cleaned-pa-train.csv")
val = pd.read_csv("/content/cleaned-pa-val.csv")

merged = pd.concat([train, val], ignore_index=True)

# Shuffle the rows
merged = merged.sample(frac=1).reset_index(drop=True)

merged.to_csv("combined.csv", index=False)

merged = pd.concat([train, val], ignore_index=True)


In [ ]:
#  Import essentials
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast

df=pd.read_csv("/content/combined.csv")


df.head()

,sentiment,translated_text
0,negative,ਧਰਮ ਅਤੇ ਕਈ ਤਰ੍ਹਾਂ ਦੇ ਵਿਤਕਰੇ ਅਤੇ ਵਿਸ਼ਵਾਸਾਂ ਦੇ ਆ...
1,positive,ਸੈਰ-ਸਪਾਟਾ ਸ਼ਹਿਰ ਮਨਾਲੀ ਘਰੇਲੂ ਅਤੇ ਵਿਦੇਸ਼ੀ ਸੈਲਾਨੀ...
2,neutral,ਕੰਪਨੀ ਨੇ ਇਸ ਡਿਵਾਈਸ 'ਚ M-70 Intel ਕੋਰ ਪ੍ਰੋਸੈਸਰ ...
3,negative,ਦੇਵ ਅਤੇ ਦਿਸ਼ਾ ਵਿਚਕਾਰ ਭਾਵੁਕ ਸੀਨ ਛੋਟੇ ਹਨ।
4,neutral,"ਟੈਬਲੇਟ ਵਿੱਚ 1.2 GHz Xburst ਪ੍ਰੋਸੈਸਰ ਹੈ, ਜੋ ਪ੍ਰ..."


In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
positive,3266
neutral,2845
negative,1408


In [ ]:
# balancing dataset
from sklearn.utils import resample

# Separate classes
df_neu = df[df['sentiment'] == 'neutral']
df_pos = df[df['sentiment'] == 'positive']
df_neg = df[df['sentiment'] == 'negative']

# Find majority class size
max_size = max(len(df_neu), len(df_pos), len(df_neg))

# Oversample minority classes
df_pos_upsampled = resample(df_pos, replace=True, n_samples=max_size, random_state=42)
df_neg_upsampled = resample(df_neg, replace=True, n_samples=max_size, random_state=42)
df_neu_upsampled = resample(df_neu, replace=True, n_samples=max_size, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([df_neu_upsampled, df_pos_upsampled, df_neg_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced['sentiment'].value_counts())

sentiment
neutral     3266
negative    3266
positive    3266
Name: count, dtype: int64


In [ ]:
# Map labels to integers
label_map = {'positive': 0, 'negative': 1, 'neutral': 2}
df_balanced['label'] = df_balanced['sentiment'].map(label_map)

# Shuffle before splitting
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Rename column
df= df.rename(columns={"translated_text": "text"})


df = df[['text', 'label']]
print(df.head())

                                                text  label
0  ਇਸ ਫੋਨ ਦੀ ਸਭ ਤੋਂ ਚੰਗੀ ਗੱਲ ਇਹ ਹੋਵੇਗੀ ਕਿ ਰੈੱਡਮੀ ...      0
1                           ਗ੍ਰਾਫਿਕ ਗੁਣਵੱਤਾ ਚੰਗੀ ਹੈ.      0
2  ਇਸਦੀ ਲੰਬੇ ਸਮੇਂ ਤੱਕ ਵਰਤੋਂ ਡਿਵਾਈਸ ਨੂੰ ਗਰਮ ਹੋਣ ਦੀ...      0
3  ਫਿਲਮ ਨੂੰ ਦੇਖਦੇ ਹੋਏ ਲੱਗਦਾ ਹੈ ਕਿ ਗੀਤਾਂ ਦੇ ਮਿਊਜ਼ਿ...      2
4  ਮਿਥੁਨ ਪੂਰੀ ਫਿਲਮ 'ਚ ਦਮਦਾਰ ਨਜ਼ਰ ਆ ਰਹੇ ਹਨ। ਬੱਚਿਆਂ...      0


In [ ]:
# Step: Tokenization using Hugging Face Datasets + Tokenizer
# Install first if needed: !pip install transformers datasets

from datasets import Dataset
from transformers import AutoTokenizer

In [ ]:
# Use multilingual BERT (best for Urdu/Roman Urdu + English)
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [ ]:
# Split into train (80%), validation (10%), test (10%)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

# Print sizes
print("✅ Train:", len(train_df), " Validation:", len(val_df), " Test:", len(test_df))
print(train_df['label'].value_counts())

✅ Train: 7838  Validation: 980  Test: 980
label
0    2613
1    2613
2    2612
Name: count, dtype: int64


In [ ]:
# Convert pandas DataFrames to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

In [ ]:
# Tokenize function
MAX_LEN = 128  # adjust if your sentences are long
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",   # ensures uniform length
        truncation=True,
        max_length=MAX_LEN
    )


In [ ]:
# Apply tokenization
train_ds = train_ds.map(tokenize_function, batched=True)
val_ds   = val_ds.map(tokenize_function, batched=True)
test_ds  = test_ds.map(tokenize_function, batched=True)

Map:   0%|          | 0/7838 [00:00<?, ? examples/s]

Map:   0%|          | 0/980 [00:00<?, ? examples/s]

Map:   0%|          | 0/980 [00:00<?, ? examples/s]

In [ ]:
# Set format for PyTorch
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("✅ Tokenization done.")
print(train_ds[0])

✅ Tokenization done.
{'label': tensor(0), 'input_ids': tensor([   101,   1007,  23415,  45378,  58125,  68614,  15982,  13766,  16154,
          1023,  45378,  68769,   1009,  13475,  75919,  33122,  12666,   1021,
        111256, 111266,  31948,  13094,   1044,  45378,  42798,  13094,   1019,
         42792,  39503,  22236,  11770,    920,    102,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,  

In [ ]:
pip install transformers datasets peft accelerate bitsandbytes evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00


# LORA


In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizerFast, TrainingArguments, Trainer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import evaluate

# ---------------------------
# 1. Load tokenizer and model
# ---------------------------
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizerFast.from_pretrained(model_name)


model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,           # <-- change for your dataset
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizerFast, TrainingArguments, Trainer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import evaluate

# ---------------------------
# 1. Load tokenizer and model
# ---------------------------
# Re-initialize the model to ensure it's a fresh, un-PEFT-modified base model
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizerFast.from_pretrained(model_name)

model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,           # <-- change for your dataset
    device_map="auto"
)

# 2. Prepare LoRA configuration
# ---------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["query", "key", "value", "output.dense"]  # strongest effect
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ---------------------------
# 3. Load dataset
# ---------------------------
# The dataset loading and preparation has been done in previous cells.
# Use the already prepared datasets from previous steps.
train_set = train_ds
test_set = test_ds

# ---------------------------
# 4. Tokenization
# ---------------------------
# Tokenization is already handled in previous cells and train_ds, test_ds are ready.

# ---------------------------
# 5. Metric — Accuracy
# ---------------------------
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# ---------------------------
# 6. Training Arguments

training_args = TrainingArguments(
    output_dir="./mb_lora",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,               # LoRA needs higher LR
    num_train_epochs=15,

    fp16=True,

    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,

    weight_decay=0.01,                # ★ improves generalization
    warmup_ratio=0.1,                 # ★ stabilizes training
    lr_scheduler_type="cosine",       # ★ smoother LR → better accuracy

    logging_steps=50,
    seed=42,

    report_to="wandb",                # optional: enable W&B logging
    run_name="bert_lora_optimized"
)

# ---------------------------
# 7. Trainer
# ---------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=test_set,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ---------------------------
# 8. Train
# ---------------------------
trainer.train()

# ---------------------------
# 9. Evaluate
# ---------------------------
results = trainer.evaluate()
print("Final Accuracy:", results["eval_accuracy"])

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,919,235 || all params: 179,774,982 || trainable%: 1.0676


/tmp/ipython-input-3607128426.py:91: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adeelahmed4868 (adeelahmed4868-riphah-international-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.828200,0.793213,0.669388
2,0.722100,0.614746,0.763265
3,0.591500,0.530731,0.795918
4,0.432600,0.476239,0.839796
5,0.389300,0.411981,0.856122
6,0.331600,0.437422,0.876531
7,0.251500,0.484616,0.887755
8,0.219000,0.452456,0.886735
9,0.146900,0.486756,0.893878
10,0.127400,0.490312,0.893878


In [ ]:
trainer.predict(test_ds)

PredictionOutput(predictions=array([[ 2.6035156 , -2.3808594 , -0.6743164 ],
       [-1.9736328 ,  4.234375  , -1.8447266 ],
       [ 1.0341797 , -4.0585938 ,  2.6582031 ],
       ...,
       [-0.25146484, -3.515625  ,  3.6054688 ],
       [ 3.1894531 , -2.8847656 , -0.79003906],
       [ 4.2929688 , -3.6464844 , -1.3544922 ]], dtype=float32), label_ids=array([0, 1, 2, 1, 2, 1, 0, 1, 0, 2, 0, 2, 1, 2, 2, 0, 2, 2, 2, 1, 2, 1,
       1, 1, 1, 0, 1, 2, 0, 0, 1, 0, 2, 0, 1, 2, 1, 0, 1, 0, 1, 1, 2, 0,
       0, 2, 1, 0, 0, 2, 0, 1, 0, 1, 2, 1, 0, 2, 1, 0, 1, 0, 1, 0, 2, 0,
       1, 1, 2, 0, 0, 2, 0, 0, 0, 1, 1, 2, 2, 2, 2, 0, 1, 2, 0, 2, 2, 1,
       0, 2, 2, 1, 2, 0, 1, 0, 0, 0, 2, 1, 2, 2, 0, 0, 2, 1, 0, 2, 2, 1,
       1, 1, 1, 0, 0, 0, 1, 2, 2, 0, 2, 0, 1, 1, 1, 2, 2, 0, 1, 2, 0, 2,
       1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 2, 2, 1, 0, 1, 1, 0, 1, 0, 0,
       2, 2, 1, 0, 2, 1, 2, 2, 1, 2, 0, 1, 0, 2, 2, 1, 0, 1, 2, 2, 2, 0,
       2, 2, 1, 1, 1, 2, 2, 0, 0, 1, 0, 1, 1, 2, 2, 2, 2, 0,

# Full Fine Tuning

In [ ]:
# Install first if needed: !pip install transformers evaluate accelerate
!pip install evaluate

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
import torch

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    # Evaluation + saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # Training settings
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,            # standard for full BERT fine-tuning
    warmup_ratio=0.1,              # improves accuracy & stabilizes training
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),

    # Logging
    logging_dir="./logs",
    logging_steps=50,

    # Enable Weights & Biases
    report_to="wandb",             # <--- ENABLE W&B
    run_name="bert_full_finetune", # your run name

    # Reproducibility
    seed=42,
)


In [ ]:
# 4️⃣ Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-705311372.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [ ]:
# 5️⃣ Train
trainer.train()
# 9. Evaluate
# ---------------------------
results = trainer.evaluate()
print("Final Accuracy:", results["eval_accuracy"])

Epoch,Training Loss,Validation Loss,Accuracy
1,0.238600,0.445648,0.859184
2,0.208500,0.449374,0.870408
3,0.228300,0.460044,0.871429
4,0.214800,0.490760,0.873469
5,0.190300,0.485993,0.874490
6,0.206300,0.493357,0.881633
7,0.148700,0.496158,0.883673
8,0.184900,0.502866,0.874490
9,0.184700,0.495122,0.877551
10,0.172700,0.498775,0.875510


Final Accuracy: 0.8836734693877552


In [ ]:
# Training arguments (tuned for accuracy + low data)
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",     # evaluate each epoch
    save_strategy="epoch",           # save best each epoch
    load_best_model_at_end=True,     # restore best model
    metric_for_best_model="accuracy",
    greater_is_better=True,
    num_train_epochs=12,              # small dataset → fewer epochs
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,              # standard for BERT fine-tuning
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    seed=42,
    fp16=torch.cuda.is_available()  # automatic mixed precision
)
# ---------------------------
# 6. Trainer
# ---------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ---------------------------
# 7. Train
# ---------------------------
trainer.train()

# ---------------------------
# 8. Evaluate on test set
# ---------------------------
results = trainer.evaluate(test_set)
print("Final Test Accuracy:", results["eval_accuracy"])

/tmp/ipython-input-1881559116.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.098500,0.623441,0.890816
2,0.128300,0.614651,0.888776
3,0.131800,0.679713,0.891837
4,0.060200,0.700884,0.892857
5,0.070100,0.741446,0.894898
6,0.014000,0.743729,0.892857
7,0.058600,0.740091,0.896939
8,0.004300,0.766284,0.896939
9,0.008700,0.776103,0.903061
10,0.059400,0.769024,0.900000


Final Test Accuracy: 0.9051020408163265


In [ ]:
trainer.predict(test_ds)

PredictionOutput(predictions=array([[ 5.765625 , -3.7617188, -2.7226562],
       [-2.6796875,  7.8203125, -2.8359375],
       [-3.7167969, -4.4101562,  6.3554688],
       ...,
       [-3.7128906, -4.546875 ,  6.4570312],
       [ 5.7460938, -3.734375 , -2.828125 ],
       [ 5.796875 , -3.9257812, -2.6523438]], dtype=float32), label_ids=array([0, 1, 2, 1, 2, 1, 0, 1, 0, 2, 0, 2, 1, 2, 2, 0, 2, 2, 2, 1, 2, 1,
       1, 1, 1, 0, 1, 2, 0, 0, 1, 0, 2, 0, 1, 2, 1, 0, 1, 0, 1, 1, 2, 0,
       0, 2, 1, 0, 0, 2, 0, 1, 0, 1, 2, 1, 0, 2, 1, 0, 1, 0, 1, 0, 2, 0,
       1, 1, 2, 0, 0, 2, 0, 0, 0, 1, 1, 2, 2, 2, 2, 0, 1, 2, 0, 2, 2, 1,
       0, 2, 2, 1, 2, 0, 1, 0, 0, 0, 2, 1, 2, 2, 0, 0, 2, 1, 0, 2, 2, 1,
       1, 1, 1, 0, 0, 0, 1, 2, 2, 0, 2, 0, 1, 1, 1, 2, 2, 0, 1, 2, 0, 2,
       1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 2, 2, 1, 0, 1, 1, 0, 1, 0, 0,
       2, 2, 1, 0, 2, 1, 2, 2, 1, 2, 0, 1, 0, 2, 2, 1, 0, 1, 2, 2, 2, 0,
       2, 2, 1, 1, 1, 2, 2, 0, 0, 1, 0, 1, 1, 2, 2, 2, 2, 0, 0, 0, 1, 2,
     

In [ ]:
# Install first if needed: !pip install transformers evaluate accelerate
!pip install evaluate

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
import torch

In [ ]:
# 1️⃣ Load model (3 sentiment labels)
MODEL_NAME = "bert-base-multilingual-cased"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 2️⃣ Metric (accuracy)
metric = evaluate.load("accuracy")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

In [ ]:
# 3️⃣ Training arguments (tuned for accuracy + low data)
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",     # evaluate each epoch
    save_strategy="epoch",           # save best each epoch
    load_best_model_at_end=True,     # restore best model
    metric_for_best_model="accuracy",
    greater_is_better=True,
    num_train_epochs=10,              # small dataset → fewer epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,              # standard for BERT fine-tuning
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    seed=42,
    fp16=torch.cuda.is_available(),  # automatic mixed precision for speed
    report_to="none"                 # disable wandb/tensorboard
)

In [ ]:
# 4️⃣ Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-705311372.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 5️⃣ Train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.221800,0.661429,0.829592
2,0.185900,0.647576,0.853061
3,0.297000,0.435045,0.865306
4,0.193000,0.604201,0.869388
5,0.167000,0.703794,0.863265
6,0.092200,0.737605,0.883673
7,0.083600,0.762856,0.881633
8,0.037200,0.789622,0.886735
9,0.033700,0.824104,0.881633
10,0.007700,0.837807,0.879592


TrainOutput(global_step=4900, training_loss=0.13657993047213068, metrics={'train_runtime': 1027.2553, 'train_samples_per_second': 76.3, 'train_steps_per_second': 4.77, 'total_flos': 5155707420380160.0, 'train_loss': 0.13657993047213068, 'epoch': 10.0})

In [ ]:

trainer.predict(test_ds)


PredictionOutput(predictions=array([[ 5.84375  , -3.3300781, -3.1660156],
       [-3.1757812,  5.6875   , -1.9238281],
       [-3.0605469, -2.8125   ,  5.6367188],
       ...,
       [-2.8242188, -2.890625 ,  5.5351562],
       [ 5.796875 , -3.3183594, -3.1289062],
       [ 5.8046875, -3.4746094, -3.0957031]], dtype=float32), label_ids=array([0, 1, 2, 1, 2, 1, 0, 1, 0, 2, 0, 2, 1, 2, 2, 0, 2, 2, 2, 1, 2, 1,
       1, 1, 1, 0, 1, 2, 0, 0, 1, 0, 2, 0, 1, 2, 1, 0, 1, 0, 1, 1, 2, 0,
       0, 2, 1, 0, 0, 2, 0, 1, 0, 1, 2, 1, 0, 2, 1, 0, 1, 0, 1, 0, 2, 0,
       1, 1, 2, 0, 0, 2, 0, 0, 0, 1, 1, 2, 2, 2, 2, 0, 1, 2, 0, 2, 2, 1,
       0, 2, 2, 1, 2, 0, 1, 0, 0, 0, 2, 1, 2, 2, 0, 0, 2, 1, 0, 2, 2, 1,
       1, 1, 1, 0, 0, 0, 1, 2, 2, 0, 2, 0, 1, 1, 1, 2, 2, 0, 1, 2, 0, 2,
       1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 2, 2, 1, 0, 1, 1, 0, 1, 0, 0,
       2, 2, 1, 0, 2, 1, 2, 2, 1, 2, 0, 1, 0, 2, 2, 1, 0, 1, 2, 2, 2, 0,
       2, 2, 1, 1, 1, 2, 2, 0, 0, 1, 0, 1, 1, 2, 2, 2, 2, 0, 0, 0, 1, 2,
     

# Prefix Tuning (P-Tuning v2) for BERT

In [ ]:
!pip install -q transformers peft accelerate evaluate bitsandbytes datasets


In [ ]:
#  Configure Prefix Tuning
from peft import PrefixTuningConfig, get_peft_model, TaskType

# Optimized Prefix Tuning configuration
prefix_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,  # sequence classification
    num_virtual_tokens=40,       # number of prefix tokens
    prefix_projection=True,      # improves representation learning
    inference_mode=False,
    encoder_hidden_size=768      # explicitly set the hidden size for projection
)


In [ ]:
from transformers import AutoModelForSequenceClassification

# Load a fresh base model for Prefix Tuning with the correct number of labels
base_model_for_prefix_tuning = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# Apply Prefix Tuning to the fresh base model
model = get_peft_model(base_model_for_prefix_tuning, prefix_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Show trainable parameters
model.print_trainable_parameters()

trainable params: 14,797,827 || all params: 192,653,574 || trainable%: 7.6811


In [ ]:
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./results_prefix",

    # ------ Evaluation / Saving ------
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,     # SAFE now (fix shown below)
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # ------ Hyperparameters ------
    learning_rate=3e-4,              # good for prefix tuning
    num_train_epochs=12,              # prefix tuning benefits from more epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,

    # ------ Precision ------
    fp16=torch.cuda.is_available(),

    # ------ Logging / WandB ------
    logging_steps=50,
    report_to="wandb",               # ENABLE WandB
    run_name="prefix_tuning_experiment",  # WandB experiment name

    # ------ Reproducibility ------
    seed=42
)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-886978032.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 🔥 Train Prefix Tuning model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.953300,0.942802,0.551020
2,1.105600,1.099869,0.323469
3,0.886000,0.818808,0.641837
4,0.815200,0.835521,0.665306
5,0.880800,0.990361,0.541837
6,0.719400,0.695047,0.703061
7,0.699300,0.664855,0.736735
8,0.605900,0.653581,0.741837
9,0.536200,0.626589,0.767347
10,0.540100,0.561504,0.781633


TrainOutput(global_step=5880, training_loss=0.7329340349249288, metrics={'train_runtime': 545.7406, 'train_samples_per_second': 172.346, 'train_steps_per_second': 10.774, 'total_flos': 6187015550619648.0, 'train_loss': 0.7329340349249288, 'epoch': 12.0})

In [ ]:
# 8. Evaluate on test set
# ---------------------------
results = trainer.evaluate(test_set)
print("Final Test Accuracy:", results["eval_accuracy"])

Final Test Accuracy: 0.8173469387755102


In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import PrefixTuningConfig, get_peft_model, TaskType
import evaluate
import numpy as np
import torch

# Re-define MODEL_NAME for completeness in this cell
MODEL_NAME = "bert-base-multilingual-cased"

# Re-configure Prefix Tuning (copy from xwYveFmpK2Kk)
prefix_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=40,
    prefix_projection=True,
    inference_mode=False,
    encoder_hidden_size=768
)

In [ ]:


# Load a fresh base model for Prefix Tuning with the CORRECT number of labels (3)
base_model_for_prefix_tuning = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# Apply Prefix Tuning to the fresh base model
model = get_peft_model(base_model_for_prefix_tuning, prefix_config)

# Re-define Training Arguments (copy from DVTk3C4ELECe)
training_args = TrainingArguments(
    output_dir="./results_prefix",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    learning_rate=3e-4,
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="wandb",
    run_name="prefix_tuning_experiment",
    seed=42
)

# Re-load accuracy metric and define compute_metrics (copy from _umOEMqJLGlm)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

# Re-initialize trainer with the corrected model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-95600254.py:55: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# 🔥 Train Prefix Tuning model
trainer.train()

In [ ]:
# Evaluate
results = trainer.evaluate(test_ds)
print("✅ Prefix Tuning Test Results:", results)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000100,0.000000,1.000000
2,0.000000,0.000000,1.000000
3,0.000000,0.000000,1.000000
4,0.000000,0.000000,1.000000
5,0.000000,0.000000,1.000000
6,0.000000,0.000000,1.000000


✅ Prefix Tuning Test Results: {'eval_loss': 4.39882263947311e-08, 'eval_accuracy': 1.0}


# prompt tunning

In [ ]:
# Step 1: Load base model and tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PromptTuningConfig, get_peft_model

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Step 1b: Define prompt tuning configuration
prompt_config = PromptTuningConfig(
    task_type="SEQ_CLS",
    num_virtual_tokens=40,         # try 20–50 for best accuracy
    tokenizer_name_or_path=model_name,
)

In [ ]:
# Step 1b: Define prompt tuning configuration
prompt_config = PromptTuningConfig(
    task_type="SEQ_CLS",
    num_virtual_tokens=40,         # try 20–50 for best accuracy
    tokenizer_name_or_path=model_name,
)

In [ ]:
# Step 1c: Create PEFT model
peft_model = get_peft_model(model, prompt_config)
peft_model.print_trainable_parameters()

trainable params: 33,027 || all params: 177,888,774 || trainable%: 0.0186


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./prompt_tuned_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,            # 🔥 higher LR helps prompt tuning
    num_train_epochs=12,            # try 5–8 for small data
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    report_to="none"  # Ensure W&B is disabled
)

In [ ]:
import numpy as np
import evaluate

# Load accuracy metric
accuracy = evaluate.load("accuracy")

# Metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# Create Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-431711176.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
# Train the model
trainer.train()

# 8. Evaluate on test set
# ---------------------------
results = trainer.evaluate(test_set)
print("Final Test Accuracy:", results["eval_accuracy"])

Epoch,Training Loss,Validation Loss,Accuracy
1,1.088200,1.081092,0.373469
2,1.067900,1.053172,0.463265
3,1.052100,1.035506,0.462245
4,1.028500,1.026958,0.484694
5,1.034700,1.014572,0.491837
6,1.055300,1.008902,0.508163
7,1.036900,1.001198,0.512245
8,1.006300,0.996266,0.531633
9,0.985100,0.989698,0.529592
10,1.002100,0.988781,0.530612


Final Test Accuracy: 0.5255102040816326


In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(test_ds)
print("✅ Test Results:", test_results)

✅ Test Results: {'eval_loss': 1.1063616275787354, 'eval_model_preparation_time': 0.0038, 'eval_accuracy': 0.32142857142857145, 'eval_runtime': 2.1141, 'eval_samples_per_second': 463.546, 'eval_steps_per_second': 14.663}


# P-Tuning v2

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PromptEncoderConfig, get_peft_model

# Load base model & tokenizer
model_name ="bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Define P-Tuning v2 configuration
peft_config = PromptEncoderConfig(
    task_type="SEQ_CLS",
    num_virtual_tokens=40,     # can tune between 20–50
    encoder_hidden_size=768,   # same as BERT hidden size
)

# Apply PEFT configuration
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,804,803 || all params: 179,660,550 || trainable%: 1.0046


In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# ✅ Training arguments (optimized for P-Tuning v2)
training_args = TrainingArguments(
    output_dir="./p_tuning_v2_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,         # slightly higher LR for soft prompt learning
    num_train_epochs=12,         # 5–8 works best
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"  # Disable W&B
)

# ✅ Metric
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# ✅ Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/tmp/ipython-input-612392910.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# ✅ Step 4: Train and Evaluate P-Tuning v2 model

# Train
trainer.train()


In [ ]:
# Train the model
trainer.train()

# 8. Evaluate on test set
# ---------------------------
results = trainer.evaluate(test_set)
print("Final Test Accuracy:", results["eval_accuracy"])

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy
1,1.040300,1.002137,0.003800,0.510204
2,0.961300,0.937246,0.003800,0.572449
3,0.926800,0.883191,0.003800,0.590816
4,0.902200,0.878286,0.003800,0.606122
5,0.877200,0.863062,0.003800,0.619388
6,0.920200,0.853435,0.003800,0.631633
7,0.874400,0.839210,0.003800,0.636735
8,0.863200,0.829046,0.003800,0.633673
9,0.812700,0.833110,0.003800,0.634694
10,0.863700,0.828735,0.003800,0.635714


Final Test Accuracy: 0.3336734693877551


# AdaLoRA

In [ ]:
# Run in a notebook cell / terminal
!pip install -q transformers peft accelerate evaluate bitsandbytes datasets


In [ ]:
import os, random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "bert-base-multilingual-cased"   # best for Urdu + Roman Urdu + English
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from peft import AdaLoraConfig, get_peft_model, TaskType


adalora_config = AdaLoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                  # start rank (slightly larger for better accuracy)
    lora_alpha=32,
    target_modules=["query", "value"],  # adapt attention Q/V (good default for BERT)
    lora_dropout=0.05,
    inference_mode=False,
    init_r=16,             # initial rank
    target_r=6,            # final target low-rank
    tinit=100,             # iterations before adaptation starts
    tfinal=600,            # when adaptation finishes (adjust if dataset small)
    deltaT=10,              # adaptation frequency
    total_step=1000 # Placeholder value, should be calculated based on dataset size, batch size and epochs
)

adalora_model = get_peft_model(base_model, adalora_config)
adalora_model.print_trainable_parameters()

trainable params: 592,515 || all params: 178,448,286 || trainable%: 0.3320


/usr/local/lib/python3.12/dist-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(


In [ ]:
import evaluate, numpy as np
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./adalora_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    learning_rate=3e-4,            # good for adapter methods
    num_train_epochs=12,            # small dataset: 4-8
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    dataloader_pin_memory=False,   # set False to avoid 'pin_memory' warning if on CPU
    logging_steps=50,
    save_total_limit=2,
    report_to="none"               # disable automatic logging services
)

In [ ]:
from transformers import Trainer
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=adalora_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # stops if no val improvement for 2 epochs
)


/tmp/ipython-input-3692918051.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Disable wandb if accidentally enabled
import os
os.environ["WANDB_DISABLED"] = "true"

trainer.train()


Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy
1,1.091300,1.083104,0.032700,0.386735
2,0.944400,0.863749,0.032700,0.631633
3,0.866200,0.816976,0.032700,0.630612
4,0.777900,0.784031,0.032700,0.659184
5,0.763300,0.731479,0.032700,0.680612
6,0.721500,0.702919,0.032700,0.697959
7,0.718600,0.680075,0.032700,0.705102
8,0.664500,0.663797,0.032700,0.715306
9,0.578600,0.663858,0.032700,0.735714
10,0.625300,0.643342,0.032700,0.742857


TrainOutput(global_step=5880, training_loss=0.8010799703143892, metrics={'train_runtime': 719.0238, 'train_samples_per_second': 130.811, 'train_steps_per_second': 8.178, 'total_flos': 6229650963861504.0, 'train_loss': 0.8010799703143892, 'epoch': 12.0})

In [ ]:
val_results = trainer.evaluate(eval_dataset=val_ds)
print("Validation:", val_results)

test_results = trainer.evaluate(eval_dataset=test_ds)
print("Test:", test_results)


Validation: {'eval_loss': 0.6315387487411499, 'eval_model_preparation_time': 0.0327, 'eval_accuracy': 0.75, 'eval_runtime': 1.905, 'eval_samples_per_second': 514.431, 'eval_steps_per_second': 16.273, 'epoch': 12.0}
Test: {'eval_loss': 0.6346250772476196, 'eval_model_preparation_time': 0.0327, 'eval_accuracy': 0.7408163265306122, 'eval_runtime': 1.9925, 'eval_samples_per_second': 491.845, 'eval_steps_per_second': 15.558, 'epoch': 12.0}


# IA³ fine-tuning

In [ ]:
from peft import IA3Config, get_peft_model, TaskType

# ---------------------------
# 1️⃣ Define IA³ Configuration
# ---------------------------
ia3_config = IA3Config(
    task_type=TaskType.SEQ_CLS,          # sequence classification task
    target_modules=["query", "value"],   # apply IA³ adapters to attention Q/V layers
    # feedforward_modules=["intermediate.dense"],  # optional: include feed-forward layers for better adaptation
)

# ---------------------------
# 2️⃣ Apply IA³ to Base Model
# ---------------------------
ia3_model = get_peft_model(base_model, ia3_config)

# ---------------------------
# 3️⃣ Print Trainable Parameters
# ---------------------------
ia3_model.print_trainable_parameters()


trainable params: 20,739 || all params: 178,466,718 || trainable%: 0.0116


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
from peft import IA3Config, get_peft_model, TaskType

ia3_config = IA3Config(
    task_type=TaskType.SEQ_CLS,
    target_modules=["query", "value"],  # focus on attention parts for best results
    # feedforward_modules=["intermediate.dense"],  # optional for better adaptation
)

ia3_model = get_peft_model(base_model, ia3_config)
ia3_model.print_trainable_parameters()

trainable params: 20,739 || all params: 178,466,718 || trainable%: 0.0116


In [ ]:
import evaluate, numpy as np
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ia3_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    learning_rate=2e-4,              # slightly smaller than LoRA for stability
    num_train_epochs=12,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=2,
    dataloader_pin_memory=False,
    seed=42,
    report_to="none"                 # turn off W&B etc.
)

In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=ia3_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


/tmp/ipython-input-656194628.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Train the model
trainer.train()

# 8. Evaluate on test set
# ---------------------------
results = trainer.evaluate(test_set)
print("Final Test Accuracy:", results["eval_accuracy"])

Epoch,Training Loss,Validation Loss,Accuracy
1,0.570300,0.633918,0.750000
2,0.659900,0.632214,0.751020
3,0.594100,0.642479,0.746939
4,0.579500,0.640852,0.758163
5,0.586900,0.629018,0.756122
6,0.560100,0.639273,0.754082


Final Test Accuracy: 0.753061224489796


In [ ]:
val_results = trainer.evaluate(val_ds)
print("✅ Validation:", val_results)

test_results = trainer.evaluate(test_ds)
print("✅ Test:", test_results)

✅ Validation: {'eval_loss': 0.6408524513244629, 'eval_accuracy': 0.7581632653061224, 'eval_runtime': 2.231, 'eval_samples_per_second': 439.272, 'eval_steps_per_second': 13.895, 'epoch': 6.0}
✅ Test: {'eval_loss': 0.6412121653556824, 'eval_accuracy': 0.753061224489796, 'eval_runtime': 2.1725, 'eval_samples_per_second': 451.094, 'eval_steps_per_second': 14.269, 'epoch': 6.0}
